# 11b — Tabulation et interpolation du pricer Heston : pourquoi ça marche

## L'idée

Le pricer Heston exact (notebook 11) fait une intégration numérique de Fourier : environ 4 ms par appel. Dans la couverture on price l'option sur $\sim 80\,000$ trajectoires $\times 63$ pas $= 5$ millions de fois, à chaque époque d'entraînement. À 4 ms l'appel, c'est plusieurs heures. Infaisable.

La parade : le prix $C(S, v, \tau)$ est une fonction **lisse** de trois variables (une fois les paramètres du modèle fixés). On l'échantillonne une seule fois sur une grille grossière ($\sim$50 s), puis on **interpole** entre les points ($\sim$microsecondes). Ce notebook montre *pourquoi* l'interpolation linéaire est légitime ici, et quelle erreur exacte on fait.


In [ ]:
import numpy as np
from scipy.integrate import quad
from scipy.interpolate import RegularGridInterpolator, interp1d
import matplotlib.pyplot as plt

# --- pricer Heston par fonction caracteristique (verifie au notebook 11) ---
def heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T):
    out = []
    for u, b in [(0.5, kappa - rho*xi), (-0.5, kappa)]:
        d = np.sqrt((rho*xi*1j*phi - b)**2 - xi**2*(2*u*1j*phi - phi**2))
        g = (b - rho*xi*1j*phi + d)/(b - rho*xi*1j*phi - d)
        C = r*1j*phi*T + (kappa*theta/xi**2)*((b - rho*xi*1j*phi + d)*T - 2*np.log((1-g*np.exp(d*T))/(1-g)))
        D = (b - rho*xi*1j*phi + d)/xi**2 * ((1-np.exp(d*T))/(1-g*np.exp(d*T)))
        out.append(np.exp(C + D*v0 + 1j*phi*np.log(S0)))
    return out

def heston_call(S0, v0, r, kappa, theta, xi, rho, T, K):
    def integ(phi, i):
        f = heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T)[i]
        return (np.exp(-1j*phi*np.log(K))*f/(1j*phi)).real
    P1 = 0.5 + quad(integ, 1e-8, 200, args=(0,), limit=200)[0]/np.pi
    P2 = 0.5 + quad(integ, 1e-8, 200, args=(1,), limit=200)[0]/np.pi
    return S0*P1 - K*np.exp(-r*T)*P2

r, kappa, theta, xi, rho = 0.02, 2.0, 0.04, 0.3, -0.7
K = 100.0


## 1. La surface de prix est lisse

Une interpolation n'a de sens que si la fonction sous-jacente est régulière. Regardons le prix en fonction de $S$ (pour plusieurs maturités) et en fonction de $\tau$ : des courbes lisses, monotones, sans coin ni saut. C'est ce qui rend la tabulation viable.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

Sfine = np.linspace(60, 160, 120); v_ = 0.04
for tau in [0.25, 0.5, 1.0, 2.0]:
    C = [heston_call(S, v_, r, kappa, theta, xi, rho, tau, K) for S in Sfine]
    ax1.plot(Sfine, C, lw=1.8, label=f"tau={tau}")
ax1.axvline(K, color='k', lw=0.5, ls='--')
ax1.set_title("Prix du call Heston vs S (v=0.04)")
ax1.set_xlabel("S"); ax1.set_ylabel("prix"); ax1.legend()

taufine = np.linspace(0.05, 2.0, 60)
for S_ in [90, 100, 110]:
    C = [heston_call(S_, v_, r, kappa, theta, xi, rho, tau, K) for tau in taufine]
    ax2.plot(taufine, C, lw=1.8, label=f"S={S_}")
ax2.set_title("Prix vs maturite residuelle tau (v=0.04)")
ax2.set_xlabel("tau"); ax2.set_ylabel("prix"); ax2.legend()
fig.tight_layout(); plt.show()


À gauche : la fameuse forme convexe du call, qui tend vers le payoff $(S-K)^+$ quand $\tau$ diminue. À droite : le prix croît avec la maturité (plus de temps = plus de valeur temps). Tout est lisse, donc interpolable.


## 2. La seule vraie source d'erreur : la convexité

L'interpolation linéaire relie deux nœuds de grille par une **corde** (segment droit). Or le prix du call est **convexe** en $S$ (gamma positif). Pour une fonction convexe, la corde passe toujours **au-dessus** de la courbe. Donc l'interpolation **surestime** légèrement, d'un montant nul aux nœuds et maximal au milieu de chaque intervalle, là où le gamma est le plus fort (autour de la monnaie).

On le voit directement : on met 6 nœuds, on trace la corde, on la compare au prix exact.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
tau_, v_ = 1.0, 0.04

nodes = np.linspace(70, 140, 6)                                       # grille grossiere en S
Cn = np.array([heston_call(S, v_, r, kappa, theta, xi, rho, tau_, K) for S in nodes])
lin = interp1d(nodes, Cn)                                             # interpolation lineaire (corde)
Sfine = np.linspace(70, 140, 200)
Cex = np.array([heston_call(S, v_, r, kappa, theta, xi, rho, tau_, K) for S in Sfine])

ax1.plot(Sfine, Cex, 'b', lw=2, label="prix exact (courbe)")
ax1.plot(Sfine, lin(Sfine), 'r--', lw=1.5, label="interp lineaire (corde)")
ax1.plot(nodes, Cn, 'ko', ms=6, label="noeuds de grille")
ax1.set_title(f"Convexite : la corde est au-dessus (tau={tau_})")
ax1.set_xlabel("S"); ax1.set_ylabel("prix"); ax1.legend()

ax2.plot(Sfine, lin(Sfine) - Cex, 'purple', lw=1.8)
ax2.axhline(0, color='k', lw=0.5)
ax2.fill_between(Sfine, lin(Sfine) - Cex, 0, alpha=0.2, color='purple')
ax2.set_title("Erreur interp - exact  (>=0 : biais de convexite)")
ax2.set_xlabel("S"); ax2.set_ylabel("erreur")
fig.tight_layout(); plt.show()


À droite, l'erreur touche zéro à chaque nœud (là on price exactement) et bombe entre les nœuds. Les bosses sont plus hautes près de la monnaie et s'aplatissent loin (deep ITM/OTM, où le call est presque linéaire en $S$, gamma faible). C'est exactement la signature d'une erreur pilotée par le gamma. Théoriquement, cette erreur décroît comme le **carré** du pas de grille : diviser le pas par 2 divise l'erreur par 4.


## 3. Carte d'erreur sur le plan (S, v)

On reproduit la vraie grille (25 nœuds en $S$, 12 en $v$), on interpole, et on compare au prix exact sur un maillage fin. La couleur montre interp $-$ exact.


In [ ]:
Sg = np.linspace(55, 175, 25); vg = np.linspace(0.005, 0.15, 12); tau_ = 1.5
coarse = np.array([[heston_call(S, v, r, kappa, theta, xi, rho, tau_, K) for v in vg] for S in Sg])
itp = RegularGridInterpolator((Sg, vg), coarse, bounds_error=False, fill_value=None)

Sf = np.linspace(60, 170, 55); vf = np.linspace(0.008, 0.14, 50)
SS, VV = np.meshgrid(Sf, vf, indexing='ij')
approx = itp(np.c_[SS.ravel(), VV.ravel()]).reshape(SS.shape)
exact = np.array([[heston_call(S, v, r, kappa, theta, xi, rho, tau_, K) for v in vf] for S in Sf])
err = approx - exact

fig, ax = plt.subplots(figsize=(7.5, 5))
im = ax.pcolormesh(Sf, vf, err.T, shading='auto', cmap='RdBu_r',
                   vmin=-abs(err).max(), vmax=abs(err).max())
ax.plot(np.repeat(Sg, len(vg)), np.tile(vg, len(Sg)), 'k.', ms=2, alpha=0.4, label="noeuds de grille")
fig.colorbar(im, label="interp - exact")
ax.set_title(f"Carte d'erreur d'interpolation (tau={tau_})")
ax.set_xlabel("S"); ax.set_ylabel("v (variance)"); ax.legend(loc='upper left', fontsize=8)
fig.tight_layout(); plt.show()
print(f"erreur abs max = {abs(err).max():.4f}   moyenne = {abs(err).mean():.4f}")


On lit trois choses :

- **presque tout est rouge** (interp $\geq$ exact) : le biais de convexité, comme prévu ;
- les **bandes verticales** suivent le pas de la grille en $S$ (l'erreur s'annule sur chaque colonne de nœuds) ;
- l'erreur est concentrée là où le **gamma** est grand (autour de la monnaie) et s'efface en deep ITM/OTM et pour $v$ élevé (une grosse variance lisse la courbure).

L'erreur max reste $\sim 0.08$ sur des prix de plusieurs euros à plusieurs dizaines : négligeable.


## Ce qu'il faut retenir

- La tabulation transforme un pricer à 4 ms en une lecture à quelques microsecondes : c'est ce qui rend le hedger multi-instruments calculable.
- L'interpolation linéaire est légitime parce que la surface de prix est lisse ; la seule erreur systématique est un petit **biais positif de convexité** (corde au-dessus de la courbe), piloté par le gamma, nul aux nœuds.
- Cette erreur décroît en $O(h^2)$ avec le pas de grille $h$ : on choisit la finesse de grille pour que l'erreur soit négligeable devant les prix, sans payer plus cher que nécessaire.
- Argument bonus (utilisé dans le hedger) : on price avec le **même** interpolateur à l'achat et à la revente, donc les erreurs sont cohérentes et se compensent dans le P&L de couverture.
